# Scale study — does **solving** improve, or does **repair** improve, with model size?

Self-contained. `Runtime -> Run all`. Loads Qwen2.5 0.5B -> 7B and, per model, separates:
- **Solving** = accuracy of the model's *first confident* answer (its instinct).
- **Repair** = how much the *later* layers improve on that instinct (fixes minus self-sabotage).
- **Recovery / overwrite** rates (course-correction vs self-sabotage).

It prints a **VERDICT** and a plot. One model failing (e.g. 7B OOM) is skipped, the rest still conclude.

## 1 · Environment self-heal (run first)
If a prior install corrupted numpy, this says so — fix is **Runtime -> Disconnect and delete runtime**, reconnect, Run all.

In [ ]:
import sys, subprocess, os
try:
    import numpy, scipy.sparse, matplotlib  # noqa
    print('env OK — numpy', numpy.__version__)
except Exception as e:
    raise SystemExit('BROKEN ENV: '+repr(e)+'\n>>> Runtime -> Disconnect and delete runtime -> reconnect -> Run all.')


## 2 · Get the code + deps (no downgrades)

In [ ]:
import os
os.chdir('/content')
REPO='https://github.com/sinha-k-prat/small_fable.git'; BR='residual-orrery'
if not os.path.isdir('small_fable'):
    !git clone -q -b $BR $REPO
os.chdir('/content/small_fable')
!git fetch -q origin $BR && git reset --hard -q origin/$BR
!pip install -q bitsandbytes accelerate 'transformers>=4.44'
import transformers, torch
print('transformers', transformers.__version__, '| cuda', torch.cuda.is_available())


## 3 · Run the study (the whole thing)
Edit `--models` to drop 7B if you are tight on GPU/disk (e.g. `--models 0.5B 1.5B 3B`).
Bigger `--n` = more examples = steadier numbers (slower).

In [ ]:
import torch
dev_ok = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if dev_ok else 'NONE (will be slow on CPU)')
!python tools/scale_study.py --models 0.5B 1.5B 3B 7B --n 60 --delta 2.0 --out scale_study
from IPython.display import Image, display
display(Image(filename='scale_study.png'))


## 4 · (optional) download the raw numbers

In [ ]:
import json
print(json.dumps(json.load(open('scale_study.json')), indent=2))
try:
    from google.colab import files; files.download('scale_study.json'); files.download('scale_study.png')
except Exception as e:
    print('saved at', os.path.abspath('scale_study.json'))
